In [ ]:
import os
import json

In [ ]:
model_name = 'Mistral-Small-Instruct-2409'
model_name = 'Qwen2.5-7B-Instruct'
# model_name = 'Meta-Llama-3-8B-Instruct'
# model_name = 'Llama-2-13b-chat-hf'
# model_name = 'Qwen2.5-72B-Instruct'
games_dir = '/home/ikapetan/Frameworks/Projects-Master/FACT/FACT29/our_games_descriptions'
code_version = 'original_code'
# code_version = 'our_code'
ROUNDS = 25

In [ ]:
games = os.listdir(games_dir)
games = [game for game in games if game != 'base']
games_dir = [os.path.join(games_dir, game) for game in games]
games_dir = [os.path.join(game, 'output/'+code_version) for game in games_dir]
games_dir = [os.path.join(game, model_name) for game in games_dir]

In [ ]:
for game in games_dir:
    answers_files = [ os.path.join(game,filename) for filename in os.listdir(game) if filename.startswith("history")]
    successfull_games = []
    for file in answers_files:
        # read json
        answers = json.load(open(file))
        success = answers['finished_rounds'] == ROUNDS
        if success:
            successfull_games.append(file)

    print(game.split('/')[-4:], len(successfull_games))

In [ ]:
for game in games_dir:
    answers_files = [ os.path.join(game,filename) for filename in os.listdir(game) if filename.startswith("history")]
    successfull_games = []
    leaked_games = []
    failed_games = []
    for file in answers_files:
        # read json
        answers = json.load(open(file))
        success = answers['finished_rounds'] == ROUNDS
        if not success:
            failed_games.append(file)
            continue
        leak = False
        for round in answers['rounds']:
            raw_answer = round['public_answer']
            public_answer = raw_answer.lower().split()
            should_not_exist = ['plan', '<plan>', 'scratchpad', '<scratchpad>']
            # should_not_exist = ['plan', '<plan>', '</plan>', 'scratchpad', '<scratchpad>', '</scratchpad>']
            for keyword in should_not_exist:
                if keyword in public_answer:
                    leak = True
                    break
                if leak:
                    break
        if leak:
            leaked_games.append(file)
        else:
            successfull_games.append(file)
    print(game.split('/')[-4:], len(successfull_games), len(leaked_games), len(failed_games), len(answers_files))
    


In [ ]:
import pandas as pd

output_dir = '/home/ikapetan/Frameworks/Projects-Master/FACT/FACT29/our_games_descriptions/base/output/our_code_v2'
data = []

models = os.listdir(output_dir)
for model in models:
    model_dir = os.path.join(output_dir, model)
    answers_files = [os.path.join(model_dir, filename) for filename in os.listdir(model_dir) if filename.startswith("history")]
    successfull_games = []
    leaked_games = []
    failed_games = []
    for file in answers_files:
        # read json
        answers = json.load(open(file))
        success = answers['finished_rounds'] == ROUNDS
        if not success:
            failed_games.append(file)
            continue
        leak = False
        for round in answers['rounds']:
            raw_answer = round['public_answer']
            public_answer = raw_answer.lower().split()
            should_not_exist = ['plan', '<plan>', 'scratchpad', '<scratchpad>']
            # should_not_exist = ['plan', '<plan>', '</plan>', 'scratchpad', '<scratchpad>', '</scratchpad>']
            for keyword in should_not_exist:
                if keyword in public_answer:
                    leak = True
                    print(round['public_answer'])
                    print("="*30)
                    break
                if leak:
                    break
        if leak:
            leaked_games.append(file)
        else:
            successfull_games.append(file)

    data.append([model, len(successfull_games), len(leaked_games), len(failed_games), len(answers_files)])

df = pd.DataFrame(data, columns=['Model', 'Successful Games', 'Leaked Games', 'Failed Games', 'Total Files'])
print(df.to_string(index=False))
